In [1]:
# Run this first if on a fresh Colab session.
# lightgbm and xgboost are usually pre-installed on Colab but versions drift,
# so pinning them keeps results reproducible.

!pip install -q lightgbm==4.6.0 xgboost==3.2.0 shap==0.51.0 scikit-learn==1.6.1
print("Installs done.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 4.0 MB/s eta 0:00:00
Installs done.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import os
import json
from pathlib import Path
from copy import deepcopy

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, accuracy_score
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from scipy.stats import wilcoxon, spearmanr
import shap

warnings.filterwarnings("ignore")
np.random.seed(42)  # global seed for anything outside the main loop

print("All imports OK.")
print(f"  numpy     {np.__version__}")
print(f"  pandas    {pd.__version__}")
print(f"  sklearn   {__import__('sklearn').__version__}")

import xgboost, lightgbm
print(f"  xgboost   {xgboost.__version__}")
print(f"  lightgbm  {lightgbm.__version__}")
print(f"  shap      {shap.__version__}")


All imports OK.
  numpy     2.1.3
  pandas    2.2.3
  sklearn   1.6.1
  xgboost   3.2.0
  lightgbm  4.6.0
  shap      0.51.0


In [3]:
# ── Experiment configuration ─────────────────────────────────────────────────
# These are the exact seeds used in the paper. Chosen arbitrarily at the start
# and never changed after the first run (pre-registered by design).
SEEDS = [42, 123, 456, 1, 7, 13, 21, 37, 55, 99]

# Missingness rates tested. 20% is the main-claim rate; 5% and 10% are plotted
# to show the monotonic trend in Figure 1.
MISS_RATES = [0.05, 0.10, 0.20]

# Bonferroni thresholds — dataset-level (primary) and 12-comparison (strict)
ALPHA_DS = 0.05 / 3       # 0.0167 — three main datasets
ALPHA_12 = 0.05 / 12      # exact 0.0041667; tables display 0.0042

# Prespecified study-level warning threshold; not a universal fairness cutoff.
# Practical significance minimum. Even if Wilcoxon says p<0.05, a MG of 0.005
# isn't worth reporting as a real finding.
MG_MIN = 0.01

# Model names used throughout. LR is included for completeness but is NOT part
# of the main claim (TREE_MODELS are the 12 combinations the paper focuses on).
ALL_MODELS   = ["LR", "RF", "XGB", "LGB", "MLP"]
TREE_MODELS  = ["RF", "XGB", "LGB", "MLP"]
MAIN_DATASETS = ["Adult Income", "COMPAS", "Taiwan Credit"]

print("Config loaded.")
print(f"  Seeds        : {SEEDS}")
print(f"  Miss rates   : {MISS_RATES}")
print(f"  Alpha (DS)   : {ALPHA_DS:.4f}")
print(f"  Alpha (12)   : {ALPHA_12:.4f}")


Config loaded.
  Seeds        : [42, 123, 456, 1, 7, 13, 21, 37, 55, 99]
  Miss rates   : [0.05, 0.1, 0.2]
  Alpha (DS)   : 0.0167
  Alpha (12)   : 0.0042


In [4]:
# ── Dataset 1: Adult Income ──────────────────────────────────────────────────
# UCI dataset, 45222 rows, 14 features after cleaning.
# Sensitive attribute: sex. Focus group: female (g0).
# The class rate gap (female 11.4% vs male 31.2%) is a known limitation
# we address later via the matched-subsample experiment.

def load_adult():
    url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
           "adult/adult.data")
    cols = ["age","workclass","fnlwgt","education","education-num",
            "marital-status","occupation","relationship","race","sex",
            "capital-gain","capital-loss","hours-per-week","native-country","income"]

    df = pd.read_csv(url, header=None, names=cols,
                     na_values=" ?", skipinitialspace=True)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Binary target
    df["label"] = (df["income"].str.strip().str.replace(".", "", regex=False) == ">50K").astype(int)
    df.drop(columns=["income"], inplace=True)

    # Encode sex → sensitive attribute
    # female=0 (focus group g0), male=1
    df["sensitive"] = (df["sex"].str.strip() == "Male").astype(int)
    df.drop(columns=["sex"], inplace=True)

    # Categorical encoding is performed after the train/test split in
    # preprocess(), so encoder mappings are fitted on training data only.

    feature_cols = [c for c in df.columns if c not in ("label", "sensitive")]

    print(f"Adult Income: {len(df)} rows, {len(feature_cols)} features")
    print(f"  Female (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Male   (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

adult_df, adult_feats, adult_sens, adult_tgt = load_adult()


Adult Income: 32561 rows, 13 features
  Female (g0): 10771    positive rate: 0.109
  Male   (g1): 21790    positive rate: 0.306


In [5]:
# ── Dataset 2: COMPAS ────────────────────────────────────────────────────────
# ProPublica's recidivism dataset. 5915 rows after standard filtering.
# Sensitive attribute: race. Focus group: African-American (g0).
# We follow standard preprocessing from the fairness literature.
# Ethics note is in the paper — not repeating here, but please read it.

def load_compas():
    url = ("https://raw.githubusercontent.com/propublica/compas-analysis/"
           "master/compas-scores-two-years.csv")
    df = pd.read_csv(url)

    # Standard ProPublica filtering (same as Chouldechova 2017)
    df = df[
        (df["days_b_screening_arrest"] >= -30) &
        (df["days_b_screening_arrest"] <= 30) &
        (df["is_recid"] != -1) &
        (df["c_charge_degree"] != "O") &
        (df["score_text"] != "N/A")
    ].copy()
    df.reset_index(drop=True, inplace=True)

    keep = ["age", "c_charge_degree", "race", "sex",
            "priors_count", "days_b_screening_arrest",
            "juv_fel_count", "juv_misd_count", "juv_other_count",
            "c_days_from_compas", "is_recid"]
    df = df[keep].dropna()

    df["label"] = df["is_recid"].astype(int)

    # sensitive: African-American=0 (g0), Caucasian=1 (g1)
    df["sensitive"] = (~(df["race"] == "African-American")).astype(int)

    # Only keep African-American and Caucasian for binary comparison
    df = df[df["race"].isin(["African-American", "Caucasian"])].copy()
    df.reset_index(drop=True, inplace=True)

    # Categorical encoding is performed after the train/test split in
    # preprocess(), so encoder mappings are fitted on training data only.

    # Exclude race from model features because it defines the COMPAS focus group.
    # This matches the sensitive-feature exclusion used for the other datasets.
    feature_cols = [c for c in df.columns
                    if c not in ("label", "sensitive", "is_recid", "race")]

    print(f"COMPAS: {len(df)} rows, {len(feature_cols)} features")
    print(f"  AfAm  (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Cauc  (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

compas_df, compas_feats, compas_sens, compas_tgt = load_compas()


COMPAS: 5278 rows, 9 features
  AfAm  (g0): 3175    positive rate: 0.558
  Cauc  (g1): 2103    positive rate: 0.416


In [6]:
# ── Dataset 3: Taiwan Credit ─────────────────────────────────────────────────
# 30000 rows, UCI dataset ID 350.
# This one can't be fetched programmatically due to URL instability
# on the UCI server, so I'm using a manual upload fallback.
# Download from: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
# File name should be: UCI_Credit_Card.csv  (or similar)

from google.colab import files as colab_files

def load_taiwan(filepath=None):
    if filepath is None:
        print("Upload the Taiwan Credit CSV file now...")
        uploaded = colab_files.upload()
        filepath = list(uploaded.keys())[0]

    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Some versions have an 'id' column, some don't
    if "id" in df.columns:
        df.drop(columns=["id"], inplace=True)

    # The target is default.payment.next.month
    # Column name varies between versions — find it
    target_col = [c for c in df.columns
                  if "default" in c or "payment" in c][-1]
    df["label"] = df[target_col].astype(int)
    if target_col != "label":
        df.drop(columns=[target_col], inplace=True)

    # sensitive: sex column — 1=male, 2=female in original encoding
    # We map: female=0 (g0), male=1 (g1)
    sex_col = [c for c in df.columns if "sex" in c][0]
    df["sensitive"] = (df[sex_col] == 1).astype(int)  # male=1 → g1
    df.drop(columns=[sex_col], inplace=True)

    feature_cols = [c for c in df.columns if c not in ("label", "sensitive")]
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"Taiwan Credit: {len(df)} rows, {len(feature_cols)} features")
    print(f"  Female (g0): {(df.sensitive==0).sum()}  "
          f"  positive rate: {df[df.sensitive==0]['label'].mean():.3f}")
    print(f"  Male   (g1): {(df.sensitive==1).sum()}  "
          f"  positive rate: {df[df.sensitive==1]['label'].mean():.3f}")

    return df, feature_cols, "sensitive", "label"

taiwan_df, taiwan_feats, taiwan_sens, taiwan_tgt = load_taiwan()


Upload the Taiwan Credit CSV file now...


Saving UCI_Credit_Card.csv to UCI_Credit_Card.csv
Taiwan Credit: 30000 rows, 22 features
  Female (g0): 18112    positive rate: 0.208
  Male   (g1): 11888    positive rate: 0.242


In [7]:
# ── Core functions: preprocessing, stressor injection, evaluation ────────────

def preprocess(df, feats, target, seed):
    """
    Standard train/test split with stratification.
    Fit scaler on train only — no leakage.
    Preserves original row indices so subgroup labels stay aligned with
    the corresponding test labels and predictions.
    """
    X = df[feats].copy()
    y = df[target].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )

    # Fit category mappings on training data only. Test-only categories
    # receive the reserved value -1 rather than affecting the mapping.
    categorical_cols = X_tr.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        train_values = X_tr[col].astype(str)
        test_values = X_te[col].astype(str)
        encoder = LabelEncoder().fit(train_values)
        mapping = {value: code for code, value in enumerate(encoder.classes_)}
        X_tr.loc[:, col] = train_values.map(mapping).astype(float)
        X_te.loc[:, col] = test_values.map(mapping).fillna(-1).astype(float)

    scaler = MinMaxScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_tr), columns=feats, index=X_tr.index)
    X_te = pd.DataFrame(scaler.transform(X_te),     columns=feats, index=X_te.index)

    if not X_tr.index.is_unique or not X_te.index.is_unique:
        raise ValueError("Train/test row indices must be unique for subgroup alignment.")
    if X_tr.index.intersection(X_te.index).size != 0:
        raise ValueError("Train and test indices overlap.")

    return X_tr, X_te, y_tr, y_te


def aligned_test_sensitive(df, X_te, sens_col, y_te):
    """Return test-set sensitive labels in exactly the X_te/y_te row order."""
    sens_te = df.loc[X_te.index, sens_col].to_numpy()
    if len(sens_te) != len(X_te) or len(sens_te) != len(y_te):
        raise ValueError("Sensitive labels, test features, and test targets are misaligned.")
    return sens_te


def inject_mcar(X, rate, seed):
    """
    Completely At Random missingness — each value independently set to NaN
    with probability `rate`, regardless of its value or any other variable.
    Applied to test set only, after the model is already trained.
    """
    rng = np.random.RandomState(seed)
    X_out = X.copy()
    mask = rng.rand(*X_out.shape) < rate
    X_out[mask] = np.nan
    return X_out


def inject_mar(X, rate, sensitive, seed):
    """
    Constructed MAR-like probe — missingness probability for each feature is
    proportional to its Spearman correlation with the sensitive attribute.
    Used in the MAR probe supplementary experiment only; this is a constructed MAR-like stressor, not a general proof of formal MAR.

    This gives higher missingness to features that are more correlated
    with the sensitive attribute, which is a more realistic mechanism
    than MCAR in many real systems.
    """
    rng = np.random.RandomState(seed)
    X_out = X.copy()

    # Correlation of each feature with the sensitive attribute
    corr = np.array([abs(spearmanr(X[c], sensitive)[0]) for c in X.columns])
    if corr.sum() == 0:
        return inject_mcar(X, rate, seed)

    # Scale correlations so average rate = target rate
    weights = corr / corr.mean()

    for i, col in enumerate(X.columns):
        col_rate = min(rate * weights[i], 0.95)  # cap at 95%
        mask = rng.rand(len(X)) < col_rate
        X_out.loc[mask, col] = np.nan

    return X_out


def impute(X_tr, X_te, strategy="simple"):
    """
    Impute test set using statistics from the training set only.
    'simple' = median on the label-encoded, scaled feature matrix
    'knn'    = 5-nearest neighbours
    'mice'   = iterative imputer (MICE)
    """
    if strategy == "simple":
        imp = SimpleImputer(strategy="median")
    elif strategy == "knn":
        imp = KNNImputer(n_neighbors=5)
    elif strategy == "mice":
        imp = IterativeImputer(random_state=42, max_iter=10)
    else:
        raise ValueError(f"Unknown imputation strategy: {strategy}")

    imp.fit(X_tr)
    X_te_imp = pd.DataFrame(imp.transform(X_te), columns=X_te.columns, index=X_te.index)
    X_tr_imp = pd.DataFrame(imp.transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    return X_tr_imp, X_te_imp


def get_models(seed):
    """
    All five classifier families with the exact hyperparameters used in the paper.
    Nothing exotic — just the defaults that sklearn/xgb/lgb ship with,
    except we cap iterations to keep runtime manageable.
    """
    return {
        "LR" : LogisticRegression(max_iter=1000, C=1.0,
                                   solver="lbfgs", random_state=seed),
        "RF" : RandomForestClassifier(n_estimators=100, random_state=seed,
                                       n_jobs=-1),
        "XGB": XGBClassifier(n_estimators=100, eval_metric="logloss",
                              random_state=seed, verbosity=0,
                              use_label_encoder=False),
        "LGB": LGBMClassifier(n_estimators=100, random_state=seed,
                               verbosity=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=500,
                              early_stopping=True, random_state=seed),
    }


def evaluate(model, X_te, y_te, sensitive):
    """
    Returns accuracy and focus-group recall (recall for g0, i.e. sensitive==0).
    Also returns comparison-group recall (g1) for the EOG calculation.
    """
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)

    g0_mask = sensitive == 0
    g1_mask = sensitive == 1

    rec_g0 = recall_score(y_te[g0_mask], y_pred[g0_mask], zero_division=0)
    rec_g1 = recall_score(y_te[g1_mask], y_pred[g1_mask], zero_division=0)

    return {"accuracy": acc, "recall_g0": rec_g0, "recall_g1": rec_g1}


def masking_gap(m_clean, m_stressed):
    """
    MG = (recall_g0_clean - recall_g0_stressed) - (acc_clean - acc_stressed)

    Positive MG means focus-group recall fell more than aggregate accuracy.
    That's the fairness-evaluation gap we're measuring.
    """
    recall_drop = m_clean["recall_g0"] - m_stressed["recall_g0"]
    acc_drop    = m_clean["accuracy"]  - m_stressed["accuracy"]
    return recall_drop - acc_drop

def model_parameter_fingerprint(model):
    """Fingerprint learned hyperparameters to verify masking does not retrain the model."""
    params = model.get_params(deep=True)
    return repr(sorted((str(k), str(v)) for k, v in params.items()))

print("Core functions defined.")


Core functions defined.


In [8]:
# Primary datasets only: the same three datasets used in the main paper.
MAR_DATASETS = {
    "Adult Income": (adult_df, adult_feats, adult_sens, adult_tgt),
    "COMPAS": (compas_df, compas_feats, compas_sens, compas_tgt),
    "Taiwan Credit": (taiwan_df, taiwan_feats, taiwan_sens, taiwan_tgt),
}
MAR_MODELS = ["RF", "XGB", "LGB", "MLP"]  # all four primary models
MAR_RATES = [0.10, 0.20]
MAR_SEEDS = [42, 123, 456, 1, 7, 13, 21, 37, 55, 99]
print("Expanded MAR design:", len(MAR_DATASETS), "datasets ×", len(MAR_MODELS),
      "models ×", len(MAR_SEEDS), "seeds ×", len(MAR_RATES), "rates")


Expanded MAR design: 3 datasets × 4 models × 10 seeds × 2 rates


## Expanded test-time MAR sensitivity analysis

This separate run addresses the reviewer's concern about the former RF-only probe. It covers all 12 primary dataset–model combinations (three primary datasets × four primary models), using the same ten split seeds and the same training-fitted preprocessing and median imputation as the primary experiment. It does not alter the primary MCAR results.

**Mechanism correction:** the earlier function assigned a different fixed masking probability to each encoded feature column based on its correlation with the sensitive attribute, then sampled each cell independently of row values and group. That implementation is not MAR; it is heterogeneous-rate cellwise random masking. This run instead defines a transparent group-conditional MAR stress test: for each test row, the probability of masking each encoded feature cell depends on the observed focus-group label, never on the value of the feature being masked. The focus group has a higher rate, while group-specific probabilities are calibrated so the expected overall cell-missingness rate equals the requested rate. The sensitive label is used only to generate this simulated audit condition; it is not included among classifier features. This is one explicitly constructed mechanism, not a claim to cover all MAR processes.

The 20% comparisons are the main sensitivity rate. The 10% results are secondary. All comparisons are exploratory because this expanded analysis was initiated after peer review. Two-sided paired Wilcoxon tests are reported with Holm-adjusted p-values within each rate family; they are seed-level benchmark summaries, not population-level inference.


In [9]:
import pickle
from scipy.stats import wilcoxon

def inject_group_conditional_mar(X, rate, sensitive, seed):
    """Mask cells with probabilities conditioned on observed group (MAR stress test).

    p0 and p1 are calibrated so q0*p0 + q1*p1 == rate in expectation,
    where q0/q1 are the test-set proportions in groups 0/1. The masked feature
    values themselves do not determine their own missingness indicators.
    """
    sensitive = np.asarray(sensitive)
    if set(np.unique(sensitive)) != {0, 1}:
        raise ValueError("Both focus (0) and comparison (1) groups are required.")
    q0 = float(np.mean(sensitive == 0))
    q1 = 1.0 - q0
    # Deliberately elevate the focus-group missingness probability, while
    # preserving the requested overall expected cell-level rate.
    delta = 0.5 * rate
    p0 = rate + delta * q1
    p1 = rate - delta * q0
    if not (0 <= p0 <= 1 and 0 <= p1 <= 1):
        raise ValueError(f"Invalid calibrated probabilities: p0={p0}, p1={p1}")
    row_p = np.where(sensitive == 0, p0, p1)[:, None]
    rng = np.random.RandomState(int(seed))
    mask = rng.rand(*X.shape) < row_p
    X_masked = X.copy()
    X_masked[mask] = np.nan
    realized = float(mask.mean())
    by_group = {
        0: float(mask[sensitive == 0].mean()),
        1: float(mask[sensitive == 1].mean()),
    }
    return X_masked, realized, by_group, {0: p0, 1: p1}


mar_rows = []
for ds_name, (df, feats, sens_col, target_col) in MAR_DATASETS.items():
    print("\nDataset:", ds_name)
    for seed in MAR_SEEDS:
        X_tr, X_te, y_tr, y_te = preprocess(df, feats, target_col, seed)
        sens_te = aligned_test_sensitive(df, X_te, sens_col, y_te)
        for model_name in MAR_MODELS:
            model = get_models(seed)[model_name]
            model.fit(X_tr, y_tr)  # train once on clean data; freeze thereafter
            m_clean = evaluate(model, X_te, y_te, sens_te)
            fitted_state = pickle.dumps(model, protocol=pickle.HIGHEST_PROTOCOL)
            for rate in MAR_RATES:
                # MCAR comparator at the same nominal overall cell rate.
                X_mcar = inject_mcar(X_te, rate, seed=seed)
                _, X_mcar_imp = impute(X_tr, X_mcar, strategy="simple")
                m_mcar = evaluate(model, X_mcar_imp, y_te, sens_te)

                # Group-conditional MAR stressor; imputer is fitted on training data.
                X_mar, realized_mar, group_rates, group_probabilities = \
                    inject_group_conditional_mar(X_te, rate, sens_te, seed=seed)
                _, X_mar_imp = impute(X_tr, X_mar, strategy="simple")
                m_mar = evaluate(model, X_mar_imp, y_te, sens_te)
                if pickle.dumps(model, protocol=pickle.HIGHEST_PROTOCOL) != fitted_state:
                    raise AssertionError("Model state changed during test-time evaluation.")

                mar_rows.append({
                    "dataset": ds_name, "model": model_name, "seed": int(seed),
                    "rate_nominal": float(rate),
                    "rate_mcar_realized": float(np.isnan(X_mcar.to_numpy()).mean()),
                    "rate_mar_expected": float(rate),
                    "rate_mar_realized": realized_mar,
                    "p_mar_group0": group_probabilities[0],
                    "p_mar_group1": group_probabilities[1],
                    "rate_mar_group0_realized": group_rates[0],
                    "rate_mar_group1_realized": group_rates[1],
                    "mg_mcar": masking_gap(m_clean, m_mcar),
                    "mg_mar": masking_gap(m_clean, m_mar),
                    "mar_minus_mcar_mg": masking_gap(m_clean, m_mar) - masking_gap(m_clean, m_mcar),
                    "accuracy_drop_mcar": m_clean["accuracy"] - m_mcar["accuracy"],
                    "accuracy_drop_mar": m_clean["accuracy"] - m_mar["accuracy"],
                    "recall_g0_drop_mcar": m_clean["recall_g0"] - m_mcar["recall_g0"],
                    "recall_g0_drop_mar": m_clean["recall_g0"] - m_mar["recall_g0"],
                    "focus_positive_denominator": int(((sens_te == 0) & (np.asarray(y_te) == 1)).sum()),
                })
    print("  done")

mar_expanded = pd.DataFrame(mar_rows)
expected_n = len(MAR_DATASETS) * len(MAR_MODELS) * len(MAR_SEEDS) * len(MAR_RATES)
assert len(mar_expanded) == expected_n, (len(mar_expanded), expected_n)
assert mar_expanded.groupby(["dataset", "model", "rate_nominal"]).size().eq(len(MAR_SEEDS)).all()
print("Rows:", len(mar_expanded), "(expected:", expected_n, ")")
display(mar_expanded.head())



Dataset: Adult Income
  done

Dataset: COMPAS
  done

Dataset: Taiwan Credit
  done
Rows: 240 (expected: 240 )


,dataset,model,seed,rate_nominal,rate_mcar_realized,rate_mar_expected,rate_mar_realized,p_mar_group0,p_mar_group1,rate_mar_group0_realized,rate_mar_group1_realized,mg_mcar,mg_mar,mar_minus_mcar_mg,accuracy_drop_mcar,accuracy_drop_mar,recall_g0_drop_mcar,recall_g0_drop_mar,focus_positive_denominator
0,Adult Income,RF,42,0.1,0.100577,0.1,0.099892,0.133627,0.083627,0.131435,0.084533,0.022332,0.056364,0.034031,0.008394,0.007882,0.030726,0.064246,358
1,Adult Income,RF,42,0.2,0.200737,0.2,0.201257,0.267254,0.167254,0.266886,0.169301,0.073241,0.119162,0.045921,0.018937,0.017709,0.092179,0.136872,358
2,Adult Income,XGB,42,0.1,0.100577,0.1,0.099892,0.133627,0.083627,0.131435,0.084533,0.038376,0.060078,0.021703,0.009110,0.006961,0.047486,0.067039,358
3,Adult Income,XGB,42,0.2,0.200737,0.2,0.201257,0.267254,0.167254,0.266886,0.169301,0.093467,0.138643,0.045175,0.023851,0.020575,0.117318,0.159218,358
4,Adult Income,LGB,42,0.1,0.100577,0.1,0.099892,0.133627,0.083627,0.131435,0.084533,0.035612,0.076560,0.040949,0.011874,0.010032,0.047486,0.086592,358


In [10]:
# Paired seed-level summaries. Primary sensitivity family: 20% across 12 combinations.
summary_rows = []
for (ds_name, model_name, rate), sub in mar_expanded.groupby(
        ["dataset", "model", "rate_nominal"], sort=True):
    diff = sub["mar_minus_mcar_mg"].to_numpy(dtype=float)
    try:
        w_stat, p_two = wilcoxon(diff, alternative="two-sided", method="exact")
    except TypeError:  # compatibility with older SciPy
        w_stat, p_two = wilcoxon(diff, alternative="two-sided")
    except ValueError:  # all paired differences are exactly zero
        w_stat, p_two = 0.0, 1.0
    try:
        _, p_greater = wilcoxon(diff, alternative="greater", method="exact")
    except TypeError:
        _, p_greater = wilcoxon(diff, alternative="greater")
    except ValueError:  # all paired differences are exactly zero
        p_greater = 1.0
    summary_rows.append({
        "dataset": ds_name, "model": model_name, "rate_nominal": float(rate),
        "n_seeds": int(len(sub)), "mean_mg_mcar": float(sub["mg_mcar"].mean()),
        "mean_mg_mar": float(sub["mg_mar"].mean()),
        "mean_mar_minus_mcar_mg": float(diff.mean()),
        "sd_mar_minus_mcar_mg": float(np.std(diff, ddof=1)),
        "wilcoxon_statistic_two_sided": float(w_stat),
        "wilcoxon_p_two_sided": float(p_two),
        "wilcoxon_p_directional_mar_gt_mcar": float(p_greater),
        "mean_realized_mar_rate": float(sub["rate_mar_realized"].mean()),
        "mean_realized_mar_rate_group0": float(sub["rate_mar_group0_realized"].mean()),
        "mean_realized_mar_rate_group1": float(sub["rate_mar_group1_realized"].mean()),
    })

mar_expanded_stats = pd.DataFrame(summary_rows)
mar_expanded_stats["holm_p_two_sided_within_rate"] = np.nan
for rate, idx in mar_expanded_stats.groupby("rate_nominal").groups.items():
    pvals = mar_expanded_stats.loc[idx, "wilcoxon_p_two_sided"].to_numpy()
    order = np.argsort(pvals)
    ranked = (len(pvals) - np.arange(len(pvals))) * pvals[order]
    adjusted_sorted = np.minimum(1.0, np.maximum.accumulate(ranked))
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = adjusted_sorted
    mar_expanded_stats.loc[idx, "holm_p_two_sided_within_rate"] = adjusted
mar_expanded_stats["familywise_alpha_bonferroni"] = 0.05 / 12

mar_expanded.to_csv("mar_expanded_per_seed.csv", index=False)
mar_expanded_stats.to_csv("mar_expanded_statistics.csv", index=False)
print("Saved mar_expanded_per_seed.csv and mar_expanded_statistics.csv")
display(mar_expanded_stats.sort_values(["rate_nominal", "dataset", "model"]))


Saved mar_expanded_per_seed.csv and mar_expanded_statistics.csv


,dataset,model,rate_nominal,n_seeds,mean_mg_mcar,mean_mg_mar,mean_mar_minus_mcar_mg,sd_mar_minus_mcar_mg,wilcoxon_statistic_two_sided,wilcoxon_p_two_sided,wilcoxon_p_directional_mar_gt_mcar,mean_realized_mar_rate,mean_realized_mar_rate_group0,mean_realized_mar_rate_group1,holm_p_two_sided_within_rate,familywise_alpha_bonferroni
0,Adult Income,LGB,0.1,10,0.046835,0.071039,0.024204,0.010043,0.0,0.001953,0.000977,0.099987,0.132256,0.084105,0.023438,0.004167
2,Adult Income,MLP,0.1,10,0.012770,0.025122,0.012352,0.005936,0.0,0.001953,0.000977,0.099987,0.132256,0.084105,0.023438,0.004167
4,Adult Income,RF,0.1,10,0.037853,0.062112,0.024259,0.010407,0.0,0.001953,0.000977,0.099987,0.132256,0.084105,0.023438,0.004167
6,Adult Income,XGB,0.1,10,0.044256,0.071766,0.027509,0.009423,0.0,0.001953,0.000977,0.099987,0.132256,0.084105,0.023438,0.004167
8,COMPAS,LGB,0.1,10,0.003096,0.005922,0.002826,0.003337,5.0,0.019531,0.009766,0.099797,0.119532,0.069944,0.058594,0.004167
10,COMPAS,MLP,0.1,10,0.016315,0.020896,0.004581,0.005457,5.0,0.019531,0.009766,0.099797,0.119532,0.069944,0.058594,0.004167
12,COMPAS,RF,0.1,10,0.001552,0.004804,0.003253,0.003910,7.0,0.037109,0.018555,0.099797,0.119532,0.069944,0.058594,0.004167
14,COMPAS,XGB,0.1,10,-0.001163,0.002535,0.003698,0.003071,1.0,0.003906,0.001953,0.099797,0.119532,0.069944,0.023438,0.004167
16,Taiwan Credit,LGB,0.1,10,0.037446,0.045641,0.008196,0.001874,0.0,0.001953,0.000977,0.100279,0.119979,0.070161,0.023438,0.004167
18,Taiwan Credit,MLP,0.1,10,0.035703,0.044053,0.008350,0.002340,0.0,0.001953,0.000977,0.100279,0.119979,0.070161,0.023438,0.004167


## Interpretation and reporting guardrails

Use the saved outputs to update the manuscript only after reviewing the full run. Report the 20% results as a post-review exploratory sensitivity analysis with the exact implemented group-conditional mechanism. Report the 10% results as secondary. Do not describe the expanded test as preregistered, as covering all MAR mechanisms, or as independent-population inference. The analysis deliberately uses the same three datasets, four primary classifiers, ten seeds, and primary preprocessing as the main study; no primary MCAR CSV or figure is overwritten.

If the full run fails or a source dataset differs from the version used in the primary study, do not combine results silently. Record the data source/checksum and resolve the discrepancy first.
